<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h1 style="margin-top:0; color:#1d4ed8; font-size:34px;">
⏱️ 03 Early-Warning Dataset - First 25% of Course
</h1>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook builds the <b>first-quarter early-warning dataset</b> using only the information available during the first <b>25% of the course timeline</b>.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The goal is to simulate a realistic early prediction scenario where the model does not have access to full-course data. Instead, it uses early VLE engagement, assessment submissions, Arab-normalized time features, workload indicators, and selected demographic variables to predict whether a student may become <b>at risk</b>.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
🎯 Notebook Output
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; margin-top:0; color:#1e40af;"><b>This notebook performs:</b></p>

<ul style="font-size:16px; line-height:2; margin-bottom:0;">
  <li>Correlation-based feature analysis</li>
  <li>Removal of redundant or weakly useful features</li>
  <li>Selection of a reduced feature subset</li>
  <li>Saving the final selected features file as <code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">Selected_features_Q1.csv</code></li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This reduced dataset is prepared for the next modeling stage, where the system will evaluate how early student risk can be detected using only first-quarter course behavior.
</p>

</div>

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path


ARAB_SEMESTER_LENGTH = 150

base = pd.read_csv("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\archive\\data\\processed\\base_student_table.csv")
student_vle_full = pd.read_csv("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\archive\\data\\processed\\student_vle_full.csv")
assessment_full = pd.read_csv("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\archive\\data\\processed\\assessment_full.csv")

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The required processed tables were loaded successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
These tables were created in the previous merge and feature-engineering notebook, so this notebook can focus only on building the early-warning dataset.
</p>

</div>

<h2 style="color:#EA580C;">🧩 Build Early-Warning Dataset Function</h2>

This function creates an early-warning dataset using only student activity and assessment data available up to a selected course-progress cutoff.

In [14]:
def build_early_warning_dataset(cutoff, output_name):
    """
    Build an early-warning dataset using only events available up to a given course-progress cutoff.
    """

    # Filter VLE interactions up to the selected cutoff.
    vle_until_cutoff = student_vle_full[
        student_vle_full["relative_time"] <= cutoff
    ].copy()

    # Aggregate VLE engagement features up to the cutoff.
    vle_features = vle_until_cutoff.groupby(
        ["id_student", "code_module", "code_presentation"]
    ).agg(
        total_clicks_until_cutoff=("sum_click", "sum"),
        active_days_until_cutoff=("date", "nunique"),
        vle_module_presentation_length=("module_presentation_length", "first"),
        unique_sites_until_cutoff=("id_site", "nunique"),
        unique_activity_types_until_cutoff=("activity_type", "nunique")
    ).reset_index()

    # Calculate click intensity per active day.
    vle_features["clicks_per_active_day_until_cutoff"] = (
        vle_features["total_clicks_until_cutoff"] /
        vle_features["active_days_until_cutoff"]
    ).replace([np.inf, -np.inf], 0).fillna(0)

    # Calculate active-day ratio relative to course length.
    vle_features["active_days_ratio_until_cutoff"] = (
        vle_features["active_days_until_cutoff"] /
        vle_features["vle_module_presentation_length"]
    ).replace([np.inf, -np.inf], 0).fillna(0)

    # Map active-day ratio to the 150-day Arab semester scale.
    vle_features["arab_active_days_equivalent_until_cutoff"] = (
        vle_features["active_days_ratio_until_cutoff"] * ARAB_SEMESTER_LENGTH
    )

    # Build activity-type click features up to the cutoff.
    activity_features = vle_until_cutoff.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    activity_features.columns.name = None

    # Filter assessment submissions up to the selected cutoff.
    assessment_until_cutoff = assessment_full[
        assessment_full["submitted_relative_time"] <= cutoff
    ].copy()

    # Aggregate assessment performance and submission features up to the cutoff.
    assessment_features = assessment_until_cutoff.groupby(
        ["id_student", "code_module", "code_presentation"]
    ).agg(
        avg_score_until_cutoff=("score", "mean"),
        min_score_until_cutoff=("score", "min"),
        max_score_until_cutoff=("score", "max"),
        submitted_assessments_until_cutoff=("id_assessment", "count"),
        late_submissions_until_cutoff=("is_late", "sum"),
        avg_submission_delay_until_cutoff=("submission_delay", "mean"),
        avg_assessment_relative_time_until_cutoff=("assessment_relative_time", "mean"),
        avg_submitted_relative_time_until_cutoff=("submitted_relative_time", "mean"),
        avg_assessment_arab_day_until_cutoff=("assessment_arab_day", "mean"),
        avg_submitted_arab_day_until_cutoff=("submitted_arab_day", "mean"),
        avg_submission_delay_relative_until_cutoff=("submission_delay_relative", "mean"),
        avg_submission_delay_arab_days_until_cutoff=("submission_delay_arab_days", "mean")
    ).reset_index()

    # Merge VLE, activity-type, and assessment features with the base table.
    final_early_df = base.merge(
        vle_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    final_early_df = final_early_df.merge(
        activity_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    final_early_df = final_early_df.merge(
        assessment_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    # Fill missing numerical values with zero.
    numeric_cols = final_early_df.select_dtypes(include=["number"]).columns
    final_early_df[numeric_cols] = final_early_df[numeric_cols].fillna(0)

    # Fill missing deprivation-band values.
    final_early_df["imd_band"] = final_early_df["imd_band"].fillna("Unknown")

    # Create workload features.
    FULL_LOAD_OU = 120

    final_early_df["workload_ratio"] = (
        final_early_df["studied_credits"] / FULL_LOAD_OU
    ).clip(upper=2.0)

    final_early_df["workload_level"] = pd.cut(
        final_early_df["workload_ratio"],
        bins=[0, 0.5, 1.0, 1.5, 2.0],
        labels=["light", "normal", "high", "very_high"],
        include_lowest=True
    )

    # Save the generated early-warning dataset.
    from pathlib import Path

    output_path = Path(f"data/processed/{output_name}")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    final_early_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print("Shape:", final_early_df.shape)

    return final_early_df

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The function is now ready to generate early-warning datasets based on any selected course-progress cutoff.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
It keeps only VLE interactions and assessment submissions available before the cutoff, then rebuilds the student-level feature table for early prediction.
</p>

</div>

<h2 style="color:#EA580C;">⏱️ Generate the 25% Early-Warning Dataset</h2>

This section builds an early-warning dataset using only the first 25% of the course timeline.

In [15]:
df_25 = build_early_warning_dataset(
    cutoff=0.25,
    output_name="student_features_first_25.csv"
)

Saved: data\processed\student_features_first_25.csv
Shape: (32480, 55)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The early-warning dataset for the first 25% of the course was generated successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The resulting dataset contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
student-course-presentation records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">55</code> 
columns.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Compared with the full-course dataset, this version contains only the features available during the early stage of the course, making it suitable for early-risk prediction experiments.
</p>

</div>

<h2 style="color:#EA580C;">🔍 Reload and Review the 25% Dataset</h2>

This section reloads the saved 25% early-warning dataset to confirm that the file was exported correctly.

In [17]:
# Reload the saved 25% early-warning dataset.
student_features_first_25 = pd.read_csv(
    r"C:\Users\HP\OneDrive\المستندات\GitHub\Student-Leak-Radar\project\data\processed\student_features_first_25.csv")

# Preview the dataset structure.
print("Shape:", student_features_first_25.shape)
student_features_first_25.head()

Shape: (32480, 51)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,disability,...,late_submissions_until_cutoff,avg_submission_delay_until_cutoff,avg_assessment_relative_time_until_cutoff,avg_submitted_relative_time_until_cutoff,avg_assessment_arab_day_until_cutoff,avg_submitted_arab_day_until_cutoff,avg_submission_delay_relative_until_cutoff,avg_submission_delay_arab_days_until_cutoff,workload_ratio,workload_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,N,...,0.0,-1.0,0.136194,0.132463,20.429104,19.869403,-0.003731,-0.559701,2.0,very_high
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,N,...,1.0,0.5,0.136194,0.138060,20.429104,20.708955,0.001866,0.279851,0.5,light
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,Y,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,light
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,N,...,0.0,-2.5,0.136194,0.126866,20.429104,19.029851,-0.009328,-1.399254,0.5,light
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,N,...,1.0,7.0,0.070896,0.097015,10.634328,14.552239,0.026119,3.917910,0.5,light


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The saved 25% early-warning dataset was reloaded successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This confirms that the exported CSV file is available and can be used in the next analysis and modeling steps.
</p>

</div>

<h2 style="color:#EA580C;">🧾 Review 25% Dataset Columns</h2>

This section reviews the columns available in the generated 25% early-warning dataset.

In [18]:
# Display all columns in the 25% early-warning dataset.
student_features_first_25.columns

Index(['code_module', 'code_presentation', 'id_student', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
       'disability', 'final_result', 'target', 'module_presentation_length',
       'active_days_until_cutoff', 'vle_module_presentation_length',
       'unique_sites_until_cutoff', 'unique_activity_types_until_cutoff',
       'clicks_per_active_day_until_cutoff', 'active_days_ratio_until_cutoff',
       'arab_active_days_equivalent_until_cutoff', 'dataplus', 'dualpane',
       'externalquiz', 'forumng', 'glossary', 'homepage', 'htmlactivity',
       'oucollaborate', 'oucontent', 'ouelluminate', 'ouwiki', 'page',
       'questionnaire', 'quiz', 'repeatactivity', 'resource', 'sharedsubpage',
       'subpage', 'url', 'avg_score_until_cutoff',
       'submitted_assessments_until_cutoff', 'late_submissions_until_cutoff',
       'avg_submission_delay_until_cutoff',
       'avg_assessment_relative_time_until_cutoff',
       'avg_submitted_re

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 25% early-warning dataset contains demographic, course, VLE engagement, activity-type, assessment, timing, and workload features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Some full-course features such as late-course clicks or complete assessment summaries are not included because this dataset only uses information available during the first 25% of the course.
</p>

</div>

<h2 style="color:#EA580C;">✅ Final Sanity Check for the 25% Dataset</h2>

This section performs a final quick check of the dataset shape and structure before starting feature analysis and modeling.

In [6]:
# Create a working copy of the 25% early-warning dataset.
student_features_first_25 = student_features_first_25.copy()

# Display the dataset shape to confirm the number of rows and columns.
print(student_features_first_25.shape)

# Preview the first rows of the dataset.
student_features_first_25.head()

(32480, 51)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,disability,...,late_submissions_until_cutoff,avg_submission_delay_until_cutoff,avg_assessment_relative_time_until_cutoff,avg_submitted_relative_time_until_cutoff,avg_assessment_arab_day_until_cutoff,avg_submitted_arab_day_until_cutoff,avg_submission_delay_relative_until_cutoff,avg_submission_delay_arab_days_until_cutoff,workload_ratio,workload_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,N,...,0.0,-1.0,0.136194,0.132463,20.429104,19.869403,-0.003731,-0.559701,2.0,very_high
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,N,...,1.0,0.5,0.136194,0.138060,20.429104,20.708955,0.001866,0.279851,0.5,light
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,Y,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,light
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,N,...,0.0,-2.5,0.136194,0.126866,20.429104,19.029851,-0.009328,-1.399254,0.5,light
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,N,...,1.0,7.0,0.070896,0.097015,10.634328,14.552239,0.026119,3.917910,0.5,light


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 25% early-warning dataset contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
student-course-presentation records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">51</code> 
columns.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Each row represents one student attempt using only information available during the first 25% of the course timeline.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This makes the dataset suitable for early risk prediction before most of the course has been completed.
</p>

</div>

<h2 style="color:#EA580C;">🔢 Select Numeric Features for Correlation Analysis</h2>

This section selects only numerical columns from the 25% early-warning dataset and removes identifier columns that should not be used in correlation analysis.

In [19]:
# Select only numerical columns from the 25% early-warning dataset.
numeric_df_25 = student_features_first_25.select_dtypes(include=["number"]).copy()

# Remove id_student because it is an identifier, not a meaningful predictive feature.
if "id_student" in numeric_df_25.columns:
    numeric_df_25 = numeric_df_25.drop(columns=["id_student"])

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The numeric feature subset was prepared for correlation analysis.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">id_student</code> 
column was removed because it is only a student identifier and does not represent a behavioral, academic, or engagement feature.
</p>

</div>

<h2 style="color:#EA580C;">📈 Compute Feature Correlation Matrix</h2>

This section calculates the absolute Pearson correlation matrix for all numerical features in the 25% early-warning dataset.

In [20]:
# Compute the absolute Pearson correlation matrix
# for all numerical features.
corr_matrix_25 = numeric_df_25.corr().abs()

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The correlation matrix measures the strength of linear relationships between numerical features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Absolute correlation values were used so that both strong positive and strong negative relationships can be analyzed equally during feature-selection and multicollinearity analysis.
</p>

</div>

<h2 style="color:#EA580C;">🔥 Visualize Correlation Heatmap</h2>

This section visualizes the correlation matrix to identify highly related numerical features in the 25% early-warning dataset.

In [21]:
import plotly.graph_objects as go

# Create an interactive heatmap for the 25% feature correlation matrix.
fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix_25.values,
        x=corr_matrix_25.columns,
        y=corr_matrix_25.columns,
        colorscale="RdBu",
        zmin=-1,
        zmax=1
    )
)

# Customize the heatmap layout.
fig.update_layout(
    title="Correlation Heatmap - First 25%",
    width=1000,
    height=900
)

fig.show()

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The heatmap provides a visual overview of relationships between numerical features in the 25% early-warning dataset.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Darker cells indicate stronger correlations, which can help identify redundant features before model training.
</p>

</div>

<h2 style="color:#EA580C;">🧹 Remove Manually Selected Redundant Features</h2>

This section removes selected highly correlated or less useful numerical features from the 25% early-warning dataset.

In [22]:
# Define features to remove manually based on correlation analysis
# and feature relevance for early-warning modeling.
manual_drop_25 = [
    "max_score_until_cutoff",
    "min_score_until_cutoff",
    "studied_credits",
    "total_clicks_until_cutoff"
]

# Drop selected redundant features if they exist.
student_features_first_25 = student_features_first_25.drop(
    columns=manual_drop_25,
    errors="ignore"
)

# Display the updated dataset shape.
print(student_features_first_25.shape)

(32480, 51)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The selected redundant features were removed from the 25% early-warning dataset.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This helps reduce multicollinearity and keeps the feature set more focused before model training.
</p>

</div>

<h2 style="color:#EA580C;">💾 Save the Final 25% Early-Warning Dataset</h2>

This section saves the cleaned and feature-selected 25% early-warning dataset for the next modeling stages.

In [23]:
from pathlib import Path

# Create the processed-data directory if it does not exist.
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# Save the final 25% early-warning dataset.
student_features_first_25.to_csv(
    "data/processed/student_features_first_25.csv",
    index=False
)

print("student_features_first_25 saved successfully.")

student_features_first_25 saved successfully.


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final 25% early-warning dataset was saved successfully in the processed-data folder.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The dataset now contains cleaned, aggregated, and reduced features that are ready for machine-learning experiments and early-risk prediction tasks.
</p>

</div>

<h2 style="color:#EA580C;">🎯 Select Final Features for Q1 Model</h2>

This section selects the final feature subset for the first-quarter early-warning model based on correlation analysis, feature relevance, and domain knowledge.

In [24]:
# Select the final features that will be used in the Q1 early-warning model.
# These features were chosen based on correlation analysis and domain knowledge.
selected_features_q1 = [
    "avg_score_until_cutoff",
    "submitted_assessments_until_cutoff",
    "arab_active_days_equivalent_until_cutoff",
    "avg_submission_delay_arab_days_until_cutoff",
    "homepage",
    "age_band",
    "forumng",
    "unique_sites_until_cutoff",
    "unique_activity_types_until_cutoff",
    "clicks_per_active_day_until_cutoff",
    "resource",
    "subpage",
    "url",
    "oucontent",
    "quiz",
    "highest_education"
]

# Check that all selected features exist in the dataset.
missing_features = [
    column for column in selected_features_q1
    if column not in student_features_first_25.columns
]

if missing_features:
    raise ValueError(f"Missing selected features: {missing_features}")

# Create the final selected-feature dataset with the target column.
selected_df_q1 = student_features_first_25[selected_features_q1 + ["target"]]

# Save the selected Q1 features dataset.
selected_df_q1.to_csv(
    "data/processed/Selected_features_Q1.csv",
    index=False
)

print("Selected_features_Q1 saved successfully.")
print("Shape:", selected_df_q1.shape)

Selected_features_Q1 saved successfully.
Shape: (32480, 17)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final Q1 feature subset was saved successfully as 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">Selected_features_Q1.csv</code>.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The selected features combine academic performance, early engagement, Arab-normalized time features, VLE activity behavior, and demographic background.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This reduced dataset is ready to be used for the first-quarter early-warning modeling stage.
</p>

</div>

<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h2 style="margin-top:0; color:#1d4ed8; font-size:30px;">
✅ Notebook Summary
</h2>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook successfully created the <b>first-quarter early-warning dataset</b> for student risk prediction.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The workflow included loading processed intermediate tables, generating the <b>25% cutoff dataset</b>, reviewing its structure, preparing numerical features, analyzing correlations, removing redundant columns, and selecting the final Q1 feature subset.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
📦 Final Output
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; line-height:1.8; color:#334155; margin:0;">
The final output, 
<code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">Selected_features_Q1.csv</code>, 
contains the most relevant early-stage features for modeling student risk before most of the course has been completed.
</p>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This dataset will be used in the next notebook to train and evaluate the first-quarter early-warning prediction model.
</p>

</div>